# 👥 04 - Comparing Groups

Module 06 taught the mechanics of `groupby()`, `.agg()`, and `value_counts()`. This notebook uses them for the most common question in data analysis: **"Is this group different from that one?"**

Comparing groups well takes more than the right method call. You need the right **statistic** for the column type (notebook 01), a fair **denominator** (counts vs. rates), enough **data** in each group, and an eye out for a **second variable** hiding behind the first.

## ✅ Learning Goals
- Summarize a numeric column by group with center **and** spread, plus the group size
- Compare a yes/no outcome across groups as a **rate**, not a count
- Build two-way tables with `pd.crosstab()` (counts and row percentages) and `pivot_table()`
- Compare groups made from a numeric column by binning it with `pd.cut()`
- Spot small groups and confounding variables before drawing conclusions

In [1]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})

## 1. Numeric Column by Group: Center, Spread, and Size
A group mean on its own hides too much. A useful group summary always has **how many**, **where the middle is**, and **how spread out**. Named aggregation (Module 06) keeps the column names readable:

In [2]:
fare_by_class = (
    titanic
    .groupby('class')
    .agg(
        passengers=('Fare', 'size'),
        mean_fare=('Fare', 'mean'),
        median_fare=('Fare', 'median'),
        std_fare=('Fare', 'std'),
        max_fare=('Fare', 'max'),
    )
    .round(2)
)
fare_by_class

,passengers,mean_fare,median_fare,std_fare,max_fare
class,,,,,
1st,216,84.15,60.29,78.38,512.33
2nd,184,20.66,14.25,13.42,73.50
3rd,491,13.68,8.05,11.78,69.55


Read it with notebooks 02 and 03 in mind. 1st class has the biggest gap between mean and median, and by far the biggest std: the outlier tickets from notebook 03 all live in this group. To compare **typical** fares across classes, the median is the fairer choice.

### Quick Version: `.describe()` by Group
`groupby(...)[col].describe()` gives the full `.describe()` table for each group in one line:

In [3]:
titanic.groupby('class')['Age'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
class,,,,,,,,
1st,186.0,38.2,14.8,0.9,27.0,37.0,49.0,80.0
2nd,173.0,29.9,14.0,0.7,23.0,29.0,36.0,70.0
3rd,355.0,25.1,12.5,0.4,18.0,24.0,32.0,74.0


✅ **Your Turn:** Build a named-aggregation table of `Age` by `Sex` with the count of **known** ages (`'count'`), the mean, the median, and the std. Why does `'count'` differ from `'size'` here? (Module 06's `.size()` vs. `.count()`.)

In [ ]:
# Your turn here
age_by_sex = titanic.groupby('Sex').agg(
    age_count=('Age', 'count'),
    age_mean=('Age', 'mean'),
    age_median=('Age', 'median'),
    age_std=('Age', 'std')
).round(1)
age_by_sex

,age_count,age_mean,age_median,age_std
Sex,,,,
female,261,27.9,27.0,14.1
male,453,30.7,29.0,14.7


In [7]:
titanic.groupby('Sex').size()


Sex
female    314
male      577
dtype: int64

In [9]:
titanic.groupby('Sex')['Age'].count()

Sex
female    261
male      453
Name: Age, dtype: int64

size() counts all rows in each group, including rows where Age is missing (NaN).
count() counts only non-missing values in the specified column.

## 2. Yes/No Outcome by Group: Rates, Not Counts
Notebook 01 showed that the mean of a 0/1 column is a proportion. Group by something, take the mean of `Survived`, and you get a **survival rate per group**:

In [10]:
titanic.groupby('Sex')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)

,passengers,survived,survival_rate
Sex,,,
female,314,233,0.742038
male,577,109,0.188908


Look at the `survived` and `survival_rate` columns together. 233 women and 109 men survived. Comparing **counts** would already say "women did better," but only because the counts happen to agree with the rates. Counts depend on how big each group is. Rates divide that out.

Here's a case where counts mislead:

In [11]:
by_class = titanic.groupby('class')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)
by_class

,passengers,survived,survival_rate
class,,,
1st,216,136,0.629630
2nd,184,87,0.472826
3rd,491,119,0.242363


More 3rd-class passengers survived (119) than 2nd-class (87). But 3rd class was much bigger, and its survival **rate** was the lowest. "More survivors" and "better chances" are different claims.

> 🧠 Before comparing groups, ask: **is the denominator the same?** If not, compare rates or percentages.

## 3. Two Categorical Columns: `pd.crosstab()`
`pd.crosstab(rows, columns)` counts every combination of two categorical columns:

In [12]:
pd.crosstab(titanic['class'], titanic['Sex'], margins=True)

Sex,female,male,All
class,,,
1st,94,122,216
2nd,76,108,184
3rd,144,347,491
All,314,577,891


`margins=True` adds an `All` row and column of totals. Counts again have the denominator problem, so use `normalize=` to turn them into percentages:

| `normalize=` | Each … adds up to 100% | Answers |
|---|---|---|
| `'index'` | row | "Within each class, what share is female?" |
| `'columns'` | column | "Of all the women, what share was in each class?" |
| `'all'` | whole table | "What share of everyone is a 1st-class woman?" |

In [13]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='index') * 100).round(1)

Sex,female,male
class,,
1st,43.5,56.5
2nd,41.3,58.7
3rd,29.3,70.7


In [14]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='columns') * 100).round(1)

Sex,female,male
class,,
1st,29.9,21.1
2nd,24.2,18.7
3rd,45.9,60.1


Same data, two different questions, two different tables. Before running a crosstab, say the question out loud. The question decides which way to normalize.

## 4. A Statistic for Every Combination: `pivot_table()`
`crosstab` counts. `pivot_table` computes **any** statistic of a third column, for every combination of two others:

In [15]:
survival_table = titanic.pivot_table(
    values='Survived',
    index='class',
    columns='Sex',
    aggfunc='mean',
)
(survival_table * 100).round(1)

Sex,female,male
class,,
1st,96.8,36.9
2nd,92.1,15.7
3rd,50.0,13.5


This table says much more than either one-way table. Women in 1st and 2nd class almost all survived. Men in 2nd and 3rd class almost all didn't. The class gap is large for women (97% vs. 50%) and the sex gap is large in every class.

In Module 08, this exact table becomes a **heatmap**.

## 5. Groups from a Numeric Column: `pd.cut()`
To compare age groups, first turn `Age` into bands with `pd.cut()` (Module 06), then group by the bands.

In [16]:
titanic['age_group'] = pd.cut(
    titanic['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['child', 'teen', 'young adult', 'adult', 'senior'],
)

titanic.groupby('age_group', observed=True)['Survived'].agg(['size', 'mean']).round(3)

,size,mean
age_group,,
child,69,0.580
teen,70,0.429
young adult,358,0.383
adult,195,0.400
senior,22,0.227


`pd.cut()` returns an **ordered category** (notebook 01), so the groups come back in age order. `observed=True` tells pandas to show only the bands that actually have rows.

⚠️ The sizes add up to 714, not 891. Passengers with no recorded age aren't in any band. A survival rate for "children" says nothing about the 177 passengers whose age is unknown.

## 6. Comparing Fairly: Two Traps
### Trap 1: Tiny Groups
A rate from 5 people is much less trustworthy than a rate from 500. Always keep the group size next to the statistic, and be suspicious of extreme rates from small groups:

In [17]:
family = titanic.groupby('SibSp')['Survived'].agg(['size', 'mean']).round(2)
family

,size,mean
SibSp,,
0,608,0.35
1,209,0.54
2,28,0.46
3,16,0.25
4,18,0.17
5,5,0.00
8,7,0.00


The `5` and `8` rows show a 0% survival rate, but they're tiny groups (each one is essentially a single family). One family's fate isn't a pattern. A common fix is to only report groups above a minimum size, using Module 06's group filtering:

In [18]:
family[family['size'] >= 20]

,size,mean
SibSp,,
0,608,0.35
1,209,0.54
2,28,0.46


### Trap 2: A Hidden Second Variable (Confounding)
Passengers who boarded at Cherbourg (`C`) had the highest survival rate:

In [19]:
titanic.groupby('Embarked')['Survived'].agg(['size', 'mean']).round(3)

,size,mean
Embarked,,
C,168,0.554
Q,77,0.390
S,644,0.337


Did boarding at Cherbourg save lives? Check who boarded there:

In [20]:
(pd.crosstab(titanic['Embarked'], titanic['class'], normalize='index') * 100).round(1)

class,1st,2nd,3rd
Embarked,,,
C,50.6,10.1,39.3
Q,2.6,3.9,93.5
S,19.7,25.5,54.8


About half of the Cherbourg passengers were in 1st class, far more than at the other ports. Class affected survival, and class also differed by port. So the port's survival rate is partly a **class** effect in disguise. A variable like `class` that's tangled up with both the group and the outcome is a **confounder**.

The fix is to compare *within* levels of the suspected confounder:

In [21]:
(titanic.pivot_table(values='Survived', index='class', columns='Embarked', aggfunc='mean') * 100).round(1)

Embarked,C,Q,S
class,,,
1st,69.4,50.0,58.3
2nd,52.9,66.7,46.3
3rd,37.9,37.5,19.0


Within each class the port differences are smaller, though not all gone. Grouping alone can't prove what *caused* a difference. It can only show you where to look. Notebook 05 says the same thing about correlation.

---
### 📝 Summary
- Report **size, center, and spread** for every group, not just the mean.
- Compare yes/no outcomes as **rates** (`groupby(...)['col'].mean()`), not counts.
- `pd.crosstab()` counts combinations. `normalize='index'` / `'columns'` answers different questions.
- `pivot_table()` computes any statistic for every combination of two columns.
- Watch for **small groups** (unreliable rates) and **confounders** (a second variable behind the difference).

## 🏋️ Practice

### Practice 1 — Legendary vs. Not (easy)
Load `pokemon.csv` into `poke`. Compare `Total` for Legendary and non-Legendary Pokémon with a named-aggregation table showing the count, mean, median, and std.

In [22]:
poke = pd.read_csv('pokemon.csv')
# Your code here

legendary_stats = (
    poke
    .groupby("Legendary")
    .agg(
        count=("Total", "size"),
        mean_total=("Total", "mean"),
        median_total=("Total", "median"),
        std_total=("Total", "std"),
    )
    .round(2)
)

print(legendary_stats)

           count  mean_total  median_total  std_total
Legendary                                            
False        735      417.21         425.0     106.76
True          65      637.38         600.0      60.94


### Practice 2 — Legendary Rate by Generation (medium)
Find the share of Pokémon that are Legendary in each `Generation`, as a percentage. Then make a `pd.crosstab()` of `Generation` vs. `Legendary` with `margins=True`. Which generation has the highest Legendary **rate**, and is it also the one with the most Legendary Pokémon?

In [26]:
# Your code here
(pd.crosstab(poke['Generation'], poke['Legendary'], margins=True)* 100).round(1)

Legendary,False,True,All
Generation,,,
1,16000,600,16600
2,10100,500,10600
3,14200,1800,16000
4,10800,1300,12100
5,15000,1500,16500
6,7400,800,8200
All,73500,6500,80000


In [31]:
legendary_rate = (pd.crosstab(poke['Generation'], poke['Legendary'], normalize='index')* 100).round(1)
legendary_rate

Legendary,False,True
Generation,,
1,96.4,3.6
2,95.3,4.7
3,88.8,11.2
4,89.3,10.7
5,90.9,9.1
6,90.2,9.8


In [32]:
# Highest Legendary rate
print(legendary_rate.idxmax())

# Most Legendary Pokémon
legendary_counts = (
    pd.crosstab(poke["Generation"], poke["Legendary"])[True]
)
print(legendary_counts.idxmax())

Legendary
False    1
True     3
dtype: int64
3


### Practice 3 — Fair Type Rankings (harder)
Rank `Type 1` by median `Attack`, but **only for types with at least 30 Pokémon**. Show the size and median for the top 5. Then show which types the minimum-size rule removed, and why including them could have been misleading.

In [33]:
# Your code here
# Median Attack by Type 1, keeping only types with at least 30 Pokémon
type_attack = (
    poke.groupby("Type 1")
    .agg(
        size=("Attack", "size"),
        median_attack=("Attack", "median")
    )
)

top5 = (
    type_attack[type_attack["size"] >= 30]
    .sort_values("median_attack", ascending=False)
    .head(5)
)

print(top5)


        size  median_attack
Type 1                     
Dragon    32          113.5
Rock      44           95.0
Dark      31           88.0
Ground    32           85.0
Fire      52           84.5


In [34]:
removed = (
    type_attack[type_attack["size"] < 30]
    .sort_values("size")
)

print(removed)

          size  median_attack
Type 1                       
Flying       4           85.0
Fairy       17           52.0
Ice         24           67.0
Fighting    27          100.0
Steel       27           89.0
Poison      28           74.0


*Your explanation here.*

The minimum-size filter removes types with fewer than 30 Pokémon because their median Attack values are based on small samples and may not be representative of the type as a whole. Larger groups provide a more stable comparison, making the ranking more trustworthy.

## ✏️ Your Turn
Make a `pivot_table()` of the **median** `Fare` by `class` (rows) and `Embarked` (columns). In a sentence or two, explain how this table supports the confounding story from section 6.

In [35]:
# Your turn here
fare_pivot = pd.pivot_table(
    titanic,
    values="Fare",
    index="class",
    columns="Embarked",
    aggfunc="median"
).round(2)

print(fare_pivot)


Embarked      C      Q      S
class                        
1st       78.27  90.00  52.00
2nd       24.00  12.35  13.50
3rd        7.90   7.75   8.05


In [37]:
#row totals and column totals
fare_pivot = pd.pivot_table(
    titanic,
    values="Fare",
    index="class",
    columns="Embarked",
    aggfunc="median",
    margins=True
).round(2)

print(fare_pivot)

Embarked      C      Q      S    All
class                               
1st       78.27  90.00  52.00  58.69
2nd       24.00  12.35  13.50  14.25
3rd        7.90   7.75   8.05   8.05
All       29.70   7.75  13.00  14.45


In [42]:
# compare with crosstab
(pd.crosstab(
    titanic['class'],
    titanic['Embarked'],
    normalize='index'
) * 100).round(1)

Embarked,C,Q,S
class,,,
1st,39.7,0.9,59.3
2nd,9.2,1.6,89.1
3rd,13.4,14.7,71.9


*Your explanation here.*

The pivot table shows that median fares differ dramatically by passenger class, with First Class having much higher fares than Second or Third Class. The crosstab shows that the embarkation ports have different mixes of passenger classes. Therefore, part of the fare difference between embarkation ports is explained by class composition rather than the port itself. In other words, class is a potential confounding variable because it is associated with both Embarked and Fare. This is the confounding story: if one port has a much larger share of First-Class passengers, it will naturally have a higher overall median fare even if the port itself has nothing to do with ticket prices.

## 🔥 Challenge (Optional) — Differences Between Groups
"Women survived more" is vague. Make it precise. From `survival_table` (section 4), compute for each class:
1. the **difference** in survival rates (women − men), in percentage points, and
2. the **ratio** of the rates (women ÷ men).

In which class is the gap biggest by each measure? Why can the two measures disagree?

In [45]:
# Your code here
survival_table = titanic.pivot_table(
    values='Survived',
    index='class',
    columns='Sex',
    aggfunc='mean'
)

# Difference in percentage points (women - men)
difference_pp = (
    (survival_table['female'] - survival_table['male']) * 100
).round(1)

# Ratio (women / men)
ratio = (
    survival_table['female'] / survival_table['male']
).round(2)

result = pd.DataFrame({
    'female_rate': (survival_table['female'] * 100).round(1),
    'male_rate': (survival_table['male'] * 100).round(1),
    'difference_pp': difference_pp,
    'ratio': ratio
})

print(result.sort_index())

print("Largest difference:",
      difference_pp.idxmax(),
      difference_pp.max())

print("Largest ratio:",
      ratio.idxmax(),
      ratio.max())

       female_rate  male_rate  difference_pp  ratio
class                                              
1st           96.8       36.9           59.9   2.62
2nd           92.1       15.7           76.4   5.85
3rd           50.0       13.5           36.5   3.69
Largest difference: 2nd 76.4
Largest ratio: 2nd 5.85


---
✅ Next up → [Relationships and Correlation](05-Relationships_and_Correlation.ipynb): comparing two **numeric** columns, and the limits of a single correlation number.